In [912]:
from lib.config import Config
from lib.datasets.lane_dataset import LaneDataset
import matplotlib.pyplot as plt 
import albumentations as A
from pathlib import Path
cfg_path = "cfgs/laneatt_culane_resnet34_new.yml"
# cfg_path = "cfgs/laneatt_culane_resnet50.yml"
import re
cfg = Config(cfg_path)
import cv2

In [919]:
params = cfg["datasets"]["train_test"]["parameters"]

In [920]:
import math


def build_transform():
    """Build the augmentation sequence from the selected YAML dataset."""
    height, width = params['img_size']
    return A.Compose([
        A.Sequential([
            LaneDataset.build_augmentation(aug)
            for aug in (params.get('augmentations') or [])
        ], p=params.get('aug_chance', 1.0)),
        A.Resize(height=height, width=width),
    ])


def comparison_frame(frame, transform):
    augmented = transform(image=frame.copy())['image']
    height, width = augmented.shape[:2]
    original = cv2.resize(frame, (width, height))
    return cv2.hconcat([original, augmented])


def augment_image(image_file, output_file=None, show=True):
    """Return an original-left/augmented-right image; optionally save it."""
    image = cv2.imread(str(image_file))
    if image is None:
        raise ValueError(f'Cannot read image: {image_file}')
    comparison = comparison_frame(image, build_transform())
    if output_file is not None:
        output_file = Path(output_file)
        output_file.parent.mkdir(parents=True, exist_ok=True)
        if not cv2.imwrite(str(output_file), comparison):
            raise RuntimeError(f'Cannot save image: {output_file}')
    if show:
        plt.figure(figsize=(14, 5))
        plt.imshow(cv2.cvtColor(comparison, cv2.COLOR_BGR2RGB))
        plt.title('Original | Augmented')
        plt.axis('off')
        plt.show()
    return comparison


def augment_video(video_file, output_folder='all_video_output/testing', max_frames=None):
    """Save a two-panel MP4 in a new run folder and return its verified path.

    max_frames=None processes the entire input; use e.g. 200 for a preview.
    """
    if max_frames is not None and (not isinstance(max_frames, int) or max_frames < 1):
        raise ValueError('max_frames must be a positive integer or None')
    transform = build_transform()
    video = cv2.VideoCapture(str(video_file))
    writer = None
    count = 0
    try:
        if not video.isOpened():
            raise ValueError(f'Cannot open video: {video_file}')
        ok, frame = video.read()
        if not ok:
            raise ValueError(f'Cannot decode video: {video_file}')
        output_folder = Path(output_folder)
        output_folder.mkdir(parents=True, exist_ok=True)
        runs = [int(m.group(1)) for d in output_folder.iterdir()
                if d.is_dir() and (m := re.fullmatch(r'run(\d+)', d.name))]
        run_folder = output_folder / f'run{max(runs, default=0) + 1}'
        run_folder.mkdir()
        output_file = run_folder / 'output.mp4'
        fps = video.get(cv2.CAP_PROP_FPS)
        if not math.isfinite(fps) or fps <= 0:
            fps = 30.0
        comparison = comparison_frame(frame, transform)
        height, width = comparison.shape[:2]
        writer = cv2.VideoWriter(str(output_file), cv2.VideoWriter_fourcc(*'mp4v'),
                                 fps, (width, height))
        if not writer.isOpened():
            raise RuntimeError(f'Cannot open video writer: {output_file}')
        print(f'Writing: {output_file} (wait for completion before opening)')
        while True:
            writer.write(comparison)
            count += 1
            if count % 200 == 0:
                print(f'Processed {count} frames')
            if max_frames is not None and count >= max_frames:
                break
            ok, frame = video.read()
            if not ok:
                break
            comparison = comparison_frame(frame, transform)
    finally:
        video.release()
        if writer is not None:
            writer.release()

    check = cv2.VideoCapture(str(output_file))
    try:
        ok, decoded = check.read()
        saved_count = int(check.get(cv2.CAP_PROP_FRAME_COUNT))
        if not ok or decoded.shape[:2] != (height, width) or saved_count != count:
            raise RuntimeError(f'Output verification failed: {output_file}')
    finally:
        check.release()
    print(f'Saved and verified {count} frames: {output_file}')
    return output_file



In [973]:
# Image file: display original and augmented side by side.
# comparison = augment_image("/Users/amannindra/Projects/Auto/CuLaneDataset/driver_23_30frame/05151640_0419.MP4/00480.jpg", output_file="all_video_output/augmented.jpg")

# Video file: save a 200-frame preview. Omit max_frames for the whole video.
output_path = augment_video("video_input/IMG_5105.mp4", max_frames=200)


Writing: all_video_output/testing/run2/output.mp4 (wait for completion before opening)
Processed 200 frames
Saved and verified 200 frames: all_video_output/testing/run2/output.mp4
